In [6]:
import torch
import torch.nn as nn



"""
quant_head.modules

Row-wise INT8 quantized drop-in replacements for nn.Embedding and nn.Linear,
intended specifically for compressing the input embedding table and the
LM head (output projection) of a causal language model -- typically the
largest non-transformer-block parameters for models with big vocabularies.
"""
import torch
import torch.nn as nn

"""
quant_head.modules

Row-wise INT8 quantized drop-in replacements for nn.Embedding and nn.Linear,
intended specifically for compressing the input embedding table and the
LM head (output projection) of a causal language model -- typically the
largest non-transformer-block parameters for models with big vocabularies.
"""
import torch
import torch.nn as nn


class QuantizedEmbedding(nn.Module):
    """INT8 row-wise quantized drop-in replacement for nn.Embedding."""

    def __init__(self, original_embedding: nn.Embedding):
        super().__init__()
        self.num_embeddings = original_embedding.num_embeddings
        self.embedding_dim = original_embedding.embedding_dim
        self.padding_idx = original_embedding.padding_idx

        # Detect model precision (float16 on GPU, float32 on CPU)
        orig_dtype = original_embedding.weight.dtype
        weight = original_embedding.weight.data.float()

        max_vals = weight.abs().amax(dim=1, keepdim=True)
        scales = (max_vals / 127.0).clamp(min=1e-8)
        qweight = torch.round(weight / scales).clamp(-127, 127).to(torch.int8)

        self.register_buffer("qweight", qweight)
        # FIX 1: Store scales in the layer's original dtype (float32 on CPU, float16 on GPU)
        self.register_buffer("scales", scales.to(orig_dtype))

    @classmethod
    def from_float(cls, original_embedding: nn.Embedding) -> "QuantizedEmbedding":
        return cls(original_embedding)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        q_rows = self.qweight[input_ids]        # [*, embed_dim], int8
        scales = self.scales[input_ids]         # [*, 1]

        # FIX 2 & 3: Dequantize using self.scales.dtype dynamically
        target_dtype = self.scales.dtype
        return q_rows.to(target_dtype) * scales.to(target_dtype)

    def dequantize_full(self) -> torch.Tensor:
        """Materialize the full fp16/fp32 table. Used internally when a tied
        LM head needs to reuse this table's weights."""
        # FIX 4: Dynamically match scale dtype instead of hardcoded .half()
        return self.qweight.to(self.scales.dtype) * self.scales

    @property
    def weight_memory_bytes(self) -> int:
        return self.qweight.numel() * 1 + self.scales.numel() * 2

    def extra_repr(self) -> str:
        return f"num_embeddings={self.num_embeddings}, embedding_dim={self.embedding_dim}, dtype=int8"


class QuantizedLinear(nn.Module):
    """INT8 row-wise quantized drop-in replacement for nn.Linear."""

    def __init__(self, original_linear: nn.Linear):
        super().__init__()
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features

        orig_dtype = original_linear.weight.dtype
        weight = original_linear.weight.data.float()

        max_vals = weight.abs().amax(dim=1, keepdim=True)
        scales = (max_vals / 127.0).clamp(min=1e-8)
        qweight = torch.round(weight / scales).clamp(-127, 127).to(torch.int8)

        self.register_buffer("qweight", qweight)
        self.register_buffer("scales", scales.to(orig_dtype))

        self.bias = None
        if original_linear.bias is not None:
            self.bias = nn.Parameter(original_linear.bias.data.clone())

    @classmethod
    def from_float(cls, original_linear: nn.Linear) -> "QuantizedLinear":
        return cls(original_linear)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Dynamically match input tensor 'x' dtype
        weight = self.qweight.to(x.dtype) * self.scales.to(x.dtype)
        bias = self.bias.to(x.dtype) if self.bias is not None else None
        return nn.functional.linear(x, weight, bias)

    @property
    def weight_memory_bytes(self) -> int:
        return self.qweight.numel() * 1 + self.scales.numel() * 2

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}, dtype=int8"

In [5]:
"""
quant_head.utils
"""

from dataclasses import dataclass

import torch.nn as nn



@dataclass
class QuantizationReport:
    quantized_embeddings: bool
    quantized_head: bool
    tied_weights: bool
    head_shares_embedding_table: bool
    tied_forced_both_sides: bool = False
    embedding_memory_bytes: int = 0
    head_memory_bytes: int = 0
    baseline_memory_bytes: int=0
    
    @property
    def total_quantized_bytes(self) -> int:
        """Total memory consumed by quantized layers."""
        if self.head_shares_embedding_table:
            return self.embedding_memory_bytes
        return self.embedding_memory_bytes + self.head_memory_bytes

    @property
    def memory_saved_bytes(self) -> int:
        """Memory saved compared to original baseline."""
        return max(0, self.baseline_memory_bytes - self.total_quantized_bytes)

    @property
    def reduction_percentage(self) -> float:
        """Percentage of memory saved relative to baseline."""
        if self.baseline_memory_bytes == 0:
            return 0.0
        return (self.memory_saved_bytes / self.baseline_memory_bytes) * 100.0

    def print_summary(self):
        """Prints a clean executive summary for quickstart and user scripts."""
        print("\n" + "=" * 55)
        print("             QUANT-HEAD MEMORY REPORT             ")
        print("=" * 55)
        print(f"  Tied Weights                 : {self.tied_weights}")
        print(f"  Quantized Embeddings         : {self.quantized_embeddings}")
        print(f"  Quantized Head               : {self.quantized_head}")
        print(f"  Head Reuses Embedding Table  : {self.head_shares_embedding_table}")
        print("-" * 55)
        print(f"  Baseline Weight Memory       : {self.baseline_memory_bytes / 1e6:.2f} MB")
        print(f"  Quantized Weight Memory      : {self.total_quantized_bytes / 1e6:.2f} MB")
        print(f"  Memory Saved                 : {self.memory_saved_bytes / 1e6:.2f} MB ({self.reduction_percentage:.1f}% reduction)")
        print("=" * 55 + "\n")

def _make_shared_head(quantized_embedding: QuantizedEmbedding) -> QuantizedLinear:
    head = QuantizedLinear.__new__(QuantizedLinear)
    nn.Module.__init__(head)
    head.in_features = quantized_embedding.embedding_dim
    head.out_features = quantized_embedding.num_embeddings
    head.register_buffer("qweight", quantized_embedding.qweight)
    head.register_buffer("scales", quantized_embedding.scales)
    head.bias = None
    return head


def quantize_embeddings_and_head(
    model: nn.Module,
    quantize_embeddings: bool = True,
    quantize_head: bool = True,
) -> tuple[nn.Module, QuantizationReport]:
    if not (hasattr(model, "get_input_embeddings") and hasattr(model, "get_output_embeddings")):
        raise TypeError(
            "Model does not expose get_input_embeddings/get_output_embeddings; "
            "quant_head only supports transformers PreTrainedModel-style causal LMs."
        )

    input_embeddings = model.get_input_embeddings()
    output_embeddings = model.get_output_embeddings()

    config_tied = bool(getattr(getattr(model, "config", None), "tie_word_embeddings", False))
    weight_identity_tied = (
        output_embeddings is not None
        and input_embeddings is not None
        and isinstance(output_embeddings, nn.Linear)
        and output_embeddings.weight is input_embeddings.weight
    )
    tied = config_tied or weight_identity_tied


    # --- 1. CALCULATE BASELINE MEMORY HERE BEFORE REPLACING LAYERS ---
    baseline_bytes = 0

    # Read input embedding size (use hasattr instead of strict isinstance check)
    if input_embeddings is not None and hasattr(input_embeddings, "weight") and input_embeddings.weight is not None:
        baseline_bytes += input_embeddings.weight.numel() * input_embeddings.weight.element_size()

    # Read output head size (only add if untied, so we don't double count shared weights)
    if output_embeddings is not None and hasattr(output_embeddings, "weight") and output_embeddings.weight is not None:
        if not tied:
            baseline_bytes += output_embeddings.weight.numel() * output_embeddings.weight.element_size()
            if getattr(output_embeddings, "bias", None) is not None:
                baseline_bytes += output_embeddings.bias.numel() * output_embeddings.bias.element_size()

    # For a tied model, embedding and head are the SAME tensor. Quantizing
    # only one side leaves the untouched side holding a live reference to
    # the original fp16 tensor, so memory goes UP instead of down. Force
    # both sides together whenever either was requested on a tied model.
    tied_forced_both = False
    if tied and quantize_embeddings != quantize_head:
        tied_forced_both = True
        quantize_embeddings = quantize_head = (quantize_embeddings or quantize_head)

    quantized_input = None
    if quantize_embeddings and isinstance(input_embeddings, nn.Embedding):
        quantized_input = QuantizedEmbedding.from_float(input_embeddings)
        model.set_input_embeddings(quantized_input)

    quantized_head_done = False
    if quantize_head and output_embeddings is not None:
        if tied and quantized_input is not None:
            model.set_output_embeddings(_make_shared_head(quantized_input))
            quantized_head_done = True
        elif tied and quantized_input is None:
            if isinstance(output_embeddings, nn.Linear):
                model.set_output_embeddings(QuantizedLinear.from_float(output_embeddings))
                quantized_head_done = True
        elif isinstance(output_embeddings, nn.Linear):
            model.set_output_embeddings(QuantizedLinear.from_float(output_embeddings))
            quantized_head_done = True

    embed_mem = model.get_input_embeddings().weight_memory_bytes if quantized_input is not None else 0
    head_module = model.get_output_embeddings()
    head_mem = getattr(head_module, "weight_memory_bytes", 0) if quantized_head_done else 0

    report = QuantizationReport(
        quantized_embeddings=quantized_input is not None,
        quantized_head=quantized_head_done,
        tied_weights=tied,
        head_shares_embedding_table=tied and quantized_input is not None,
        tied_forced_both_sides=tied_forced_both,
        embedding_memory_bytes=embed_mem,
        head_memory_bytes=head_mem,
        baseline_memory_bytes=baseline_bytes,
    )
    return model, report

In [7]:
"""
benchmarks/benchmark.py

Standardized benchmark for the quant_head library.

Dataset & protocol: WikiText-2-raw (test split), evaluated with the
standard sliding-window perplexity method (stride = max_length = 512).
This is the same dataset and protocol used to report numbers in GPTQ,
AWQ, LLM.int8(), and the HF "Perplexity of fixed-length models" guide,
so results here are directly comparable to numbers reported elsewhere.

Models: chosen to cover different architectures, attribute-naming
conventions, and weight-tying behavior, so a passing run is real evidence
of "works on any HF causal LM" rather than "works on one model I tuned it for":

    - gpt2                                  untied embeddings, classic architecture
    - EleutherAI/pythia-160m                 GPT-NeoX style, different attribute names
    - Qwen/Qwen2.5-0.5B                      modern architecture, large (~152k) vocab
    - TinyLlama/TinyLlama-1.1B-Chat-v1.0      Llama family, TIED embeddings

Usage:
    python benchmarks/benchmark.py --model Qwen/Qwen2.5-0.5B
    python benchmarks/benchmark.py --all
    python benchmarks/benchmark.py --all --device cpu   # slower, no VRAM numbers
"""
import argparse
import gc
import json
import time
from dataclasses import dataclass, asdict

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

#from quant_head.utils import quantize_embeddings_and_head

DEFAULT_MODELS = [
    "gpt2",
    "EleutherAI/pythia-160m",
    "Qwen/Qwen2.5-0.5B",
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]

STRIDE = 512
MAX_LENGTH = 512
NUM_DOCS = 50  # WikiText-2 test documents concatenated for evaluation

VARIANTS = ["fp16_baseline", "int8_embeddings", "int8_embeddings_and_head"]


@dataclass
class BenchResult:
    model_id: str
    variant: str
    storage_vram_gb: float
    peak_inference_vram_gb: float
    perplexity: float
    perplexity_delta: float
    forward_latency_ms: float
    tied_weights: bool


def load_wikitext2_text():
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    return ds["text"]


def compute_perplexity(model, tokenizer, text_lines, device):
    model.eval()
    encodings = tokenizer("\n\n".join(text_lines[:NUM_DOCS]), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end_loc = 0
    for begin_loc in range(0, seq_len, STRIDE):
        end_loc = min(begin_loc + MAX_LENGTH, seq_len)
        trg_len = end_loc - prev_end_loc
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
        nlls.append(outputs.loss)

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    return torch.exp(torch.stack(nlls).mean()).item()


def measure_forward_latency(model, tokenizer, device, n_runs=20):
    inputs = tokenizer("The quick brown fox jumps over the lazy dog. " * 8, return_tensors="pt").to(device)
    with torch.no_grad():
        for _ in range(3):
            model(**inputs)
        if device == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(n_runs):
            model(**inputs)
        if device == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_runs * 1000


def cleanup(device):
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def run_one_variant(model_id, variant, text_lines, device, baseline_ppl=None):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cleanup(device)
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype).to(device)
    tied = bool(getattr(model.config, "tie_word_embeddings", False))

    if variant == "int8_embeddings":
        model, _ = quantize_embeddings_and_head(model, quantize_embeddings=True, quantize_head=False)
    elif variant == "int8_embeddings_and_head":
        model, _ = quantize_embeddings_and_head(model, quantize_embeddings=True, quantize_head=True)

    # Steady-state storage size: how much VRAM the model occupies at rest,
    # right after load+quantize, before any forward pass. This is the
    # number that reflects "does quantization shrink the checkpoint/resident
    # weights" -- separate from peak memory used *during* inference.
    if device == "cuda":
        torch.cuda.synchronize()
    storage_vram = torch.cuda.memory_allocated() / (1024 ** 3) if device == "cuda" else 0.0

    # Reset the peak counter *after* load+quantize so peak_inference_vram
    # isolates memory used during actual inference (forward pass
    # activations, logits tensors) -- not the transient overhead of the
    # quantization conversion itself. This separation matters most for
    # small models, where activation/logit memory can be comparable to or
    # larger than the embedding table being quantized.
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    ppl = compute_perplexity(model, tokenizer, text_lines, device)
    latency = measure_forward_latency(model, tokenizer, device)
    peak_inference_vram = torch.cuda.max_memory_allocated() / (1024 ** 3) if device == "cuda" else 0.0
    delta = 0.0 if baseline_ppl is None else ppl - baseline_ppl

    result = BenchResult(model_id, variant, round(storage_vram, 3), round(peak_inference_vram, 3),
                          round(ppl, 3), round(delta, 3), round(latency, 2), tied)
    del model
    cleanup(device)
    return result


def run_model_suite(model_id, text_lines, device):
    results = []
    baseline_ppl = None
    for variant in VARIANTS:
        print(f"  -> {model_id} [{variant}]")
        r = run_one_variant(model_id, variant, text_lines, device, baseline_ppl)
        if variant == "fp16_baseline":
            baseline_ppl = r.perplexity
        results.append(r)
    return results


def print_table(results):
    header = (f"{'Model':<38}{'Variant':<26}{'Storage(GB)':<13}{'PeakInfer(GB)':<15}"
              f"{'PPL':<8}{'ΔPPL':<8}{'Latency(ms)':<12}{'Tied'}")
    print("\n" + header)
    print("-" * len(header))
    for r in results:
        print(f"{r.model_id:<38}{r.variant:<26}{r.storage_vram_gb:<13}{r.peak_inference_vram_gb:<15}"
              f"{r.perplexity:<8}{r.perplexity_delta:<8}{r.forward_latency_ms:<12}{r.tied_weights}")


def main():
    parser = argparse.ArgumentParser(description="quant_head standardized benchmark (WikiText-2 perplexity)")
    parser.add_argument("--model", type=str, default=None, help="Single HF model id to benchmark")
    parser.add_argument("--all", action="store_true", help="Run the full 4-model validation suite")
    parser.add_argument("--device", type=str, default="cuda", choices=["cuda", "cpu"])
    parser.add_argument("--output", type=str, default="benchmark_results.json")
    args = parser.parse_args()

    if args.device == "cuda" and not torch.cuda.is_available():
        print("CUDA not available, falling back to CPU (perplexity numbers still valid, VRAM will read 0).")
        args.device = "cpu"

    text_lines = load_wikitext2_text()
    models = DEFAULT_MODELS if args.all else [args.model or "Qwen/Qwen2.5-0.5B"]

    all_results = []
    for model_id in models:
        print(f"\n=== {model_id} ===")
        all_results.extend(run_model_suite(model_id, text_lines, args.device))

    if  args.device == "cpu":
        print("\n[NOTE] Running on CPU.")
        print("  • Memory tracking (VRAM) is disabled (will display 0.0 GB).")
        print("  • Latency will be higher due to lack of CUDA hardware acceleration.")
        print("  • For peak memory savings and speedup benchmarks, run on a CUDA GPU.\n")


    print_table(all_results)

    with open(args.output, "w") as f:
        json.dump([asdict(r) for r in all_results], f, indent=2)
    print(f"\nSaved results to {args.output}")


import sys
sys.argv = ["benchmark.py", "--all"]
if __name__ == "__main__":
    main()

CUDA not available, falling back to CPU (perplexity numbers still valid, VRAM will read 0).

=== gpt2 ===
  -> gpt2 [fp16_baseline]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1331.54it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2513 > 1024). Running this sequence through the model will result in indexing errors


  -> gpt2 [int8_embeddings]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4111.25it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2513 > 1024). Running this sequence through the model will result in indexing errors


  -> gpt2 [int8_embeddings_and_head]


...

  -> TinyLlama/TinyLlama-1.1B-Chat-v1.0 [int8_embeddings]


Loading weights: 100%|██████████| 201/201 [00:03<00:00, 54.69it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2998 > 2048). Running this sequence through the model will result in indexing errors


  -> TinyLlama/TinyLlama-1.1B-Chat-v1.0 [int8_embeddings_and_head]


Loading weights: 100%|██████████| 201/201 [00:01<00:00, 144.29it/s]
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2998 > 2048). Running this sequence through the model will result in indexing errors



[NOTE] Running on CPU.
  • Memory tracking (VRAM) is disabled (will display 0.0 GB).
  • Latency will be higher due to lack of CUDA hardware acceleration.
  • For peak memory savings and speedup benchmarks, run on a CUDA GPU.


Model                                 Variant                   Storage(GB)  PeakInfer(GB)  PPL     ΔPPL    Latency(ms) Tied
----------------------------------------------------------------------------------------------------------------------------
gpt2                                  fp16_baseline             0.0          0.0            37.634  0.0     227.56      True
gpt2                                  int8_embeddings           0.0          0.0            37.769  0.135   296.26      True
gpt2                                  int8_embeddings_and_head  0.0          0.0            37.769  0.135   336.34      True
EleutherAI/pythia-160m                fp16_baseline             0.0          0.0            35.974  0.0     250.08      False
EleutherAI/pythia-16